In [ ]:
%matplotlib widget

In [ ]:
import hyperspy.api as hs
import pyxem as pxm
import matplotlib.pyplot as plt
import numpy as np

from scipy.interpolate import RegularGridInterpolator
from scipy.optimize import root

from scipy.ndimage import shift
from skimage.transform import AffineTransform, warp

from pathlib import Path

import importlib
import get_transforms
import get_masks
importlib.reload(get_transforms)
importlib.reload(get_masks)

In [ ]:
#funksjon som laster inn zspy-fil og summerer diffrkasjonsmønsteret --> brightfield-bilde
def process_stem(file_path):
    s = hs.load(file_path, lazy=True)
    s_bs = s.sum(axis=(-1, -2))
    s_bs.compute()
    return s_bs, s

#bruker funksjonen på alle zspy-filene fra 002 - 023 = 22 filer
data_folder = Path("data")
files = sorted(data_folder.glob("*.zspy"))
original_images = {}
s_dir = {}

for file in files:
    print(f"Processing {file.name}")
    s_bs, s = process_stem(file)
    original_images[file.stem] = s_bs
    s_dir[file.stem] = s

#Sjekker om filene er prosessert: 
print(f"Processed {len(original_images)} files")

In [ ]:
registered_images, transformations, shifts = get_transforms.main(original_images)

In [ ]:
masks = get_masks.main(original_images, registered_images)

In [ ]:
fig, ax = plt.subplots(1, 3)

In [ ]:
s_bs = s_dir["021"].get_direct_beam_position(method="center_of_mass")
s_bs.compute()
s_bs_lp = s_bs.get_linear_plane()
s_bs_corr = s_bs-s_bs_lp

In [ ]:
s_bs_corr_warp = warp(
    s_bs_corr.data,
    inverse_map=AffineTransform(matrix=transformations["021"]),
    preserve_range=True
)

s_bs_corr_warp_shift = shift(
    s_bs_corr_warp[:, :, 0],
    shifts["021"]
)

In [ ]:
ax[0].imshow(s_bs_corr.isig[0], cmap="grey")
ax[1].imshow(s_bs_corr_warp[:, :, 0], cmap="grey")
ax[2].imshow(s_bs_corr_warp_shift, cmap="grey")